Eliminate the first and the last five minutes data.

- Reason: During the initial minutes, patients may be adapting to the situation (e.g., feeling uncomfortable), potentially introducing additional noise.

- Butterworth Bandpass Filter (BBF) have transient effects at the beginning and end of the filtered signal.

In [1]:
## Waveform 1st
import os
import pandas as pd

directory = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/01_Waveform_Raw_248_1st/FTandBBF_243_1st/"

# Create a new directory to store modified files
new_directory = os.path.join(directory, "No55_243_1st")
os.makedirs(new_directory, exist_ok=True)

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".csv"):
        filepath = os.path.join(directory, filename)
        
        # Read the CSV file into a pandas DataFrame
        df = pd.read_csv(filepath)

        # Convert 'Time' column to datetime format
        df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%d %H:%M:%S.%f')

        # Check if the 'Time' column exists
        if 'Time' in df.columns:
            # Delete the first and last five minutes of data
            five_minutes = pd.Timedelta(minutes=5)
            min_time = df['Time'].min() + five_minutes
            max_time = df['Time'].max() - five_minutes
            df = df[(df['Time'] > min_time) & (df['Time'] < max_time)]

            # Save the modified DataFrame to a new CSV file in the new directory
            new_filepath = os.path.join(new_directory, filename)
            df.to_csv(new_filepath, index=False)

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Path to the directory containing the CSV files
dir_path = '/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/TRAPPARDS/WaveformData/01_Waveform_Raw_248_1st/FTandBBF_243_1st/No55_243_1st/'

# Path to the folder where images will be saved
image_dir = os.path.join(dir_path, 'Image')

# Create the Image directory if it doesn't exist
if not os.path.exists(image_dir):
    os.makedirs(image_dir)

# List all CSV files in the directory
csv_files = [f for f in os.listdir(dir_path) if f.endswith('.csv')]

# Helper function to determine data type from non-empty column names
def determine_data_type(df):
    for column_name in df.columns:
        if "CDGR - Flow" in column_name and df[column_name].notna().sum() > 0:
            return "CDGR", column_name
        elif "AVEA - Air Flow Wave" in column_name and df[column_name].notna().sum() > 0:
            return "AVEA", column_name
        elif "SVU - Flow" in column_name and df[column_name].notna().sum() > 0:
            return "SVU", column_name
    return None, None  # No valid non-empty column found

# Function to plot the flow columns as two subplots (one on top of the other) and save the plot
def plot_flow_columns(file_path, vent_type, flow_column, flow_ftandbbf_column):
    # Load the CSV file
    data = pd.read_csv(file_path)

    # Check if both columns are not empty
    if not data[flow_column].isna().all() and not data[flow_ftandbbf_column].isna().all():
        # Create subplots (2 rows, 1 column)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10))

        # Plot the Flow column on the first subplot
        ax1.plot(data[flow_column])
        ax1.set_title(f'{flow_column}')
        ax1.set_xlabel('Time')
        ax1.set_ylabel('Flow')

        # Plot the Flow (FTandBBF) column on the second subplot
        ax2.plot(data[flow_ftandbbf_column], color='orange')
        ax2.set_title(f'{flow_ftandbbf_column}')
        ax2.set_xlabel('Time')
        ax2.set_ylabel('Flow (FTandBBF)')

        # Set the overall plot title
        plt.suptitle(f'Flow Comparison for {vent_type} - {os.path.basename(file_path)}', fontsize=16)

        # Save the plot as an image
        base_filename = os.path.splitext(os.path.basename(file_path))[0]  # Get the base file name without extension
        image_file_path = os.path.join(image_dir, f'{base_filename}_{vent_type}_Flow_Comparison.png')
        plt.savefig(image_file_path)
        plt.close()  # Close the plot to free memory

# Loop through all CSV files
for csv_file in csv_files:
    file_path = os.path.join(dir_path, csv_file)
    
    # Load the CSV file to check the columns
    df = pd.read_csv(file_path)
    
    # Use the helper function to determine the vent type and non-empty column
    vent_type, flow_column = determine_data_type(df)
    
    # If a valid vent type and column is found, look for the FTandBBF column
    if vent_type and flow_column:
        flow_ftandbbf_column = flow_column.replace('Flow', 'Flow (FTandBBF)')  # Assume the FTandBBF column follows this naming convention
        if flow_ftandbbf_column in df.columns:
            plot_flow_columns(file_path, vent_type, flow_column, flow_ftandbbf_column)
            